In [ ]:
import mercury as mr 
from IPython.display import clear_output

In [ ]:
# welcome markdown
welcome_md = mr.Markdown("# Upload the dataset and check the summary") 

In [ ]:
input_file = mr.UploadFile(label="Upload your Dataset", accept='.csv', max_file_size='1GB')

In [ ]:
if input_file.name is not None:
    import pandas as pd
    from io import BytesIO
    from skrub import TableReport
    data = BytesIO(input_file.value)
    df = pd.read_csv(data)

In [ ]:
if input_file.name is not None:
    # shape 
    row_count = df.shape[0]
    col_count = df.shape[1]
    # mising values
    missing_count = df.isna().sum().sum()
    missing_procent = df.isna().mean().mean() * 100
    # duplicates
    duplicates_count = df.duplicated().sum()
    duplicates_procent = (duplicates_count / row_count) * 100
    # columns types
    all_columns = df.columns.tolist()
        # basic types from pandas
    numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
    boolean_columns = df.select_dtypes(include=["bool"]).columns.tolist()
    datetime_columns = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
    object_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
        # date as text detection
    detected_datetime_columns = []
    for col in object_columns:
        converted = pd.to_datetime(df[col], errors="coerce")
        valid_ratio = converted.notna().mean()
    
        if valid_ratio > 0.8:
            detected_datetime_columns.append(col)
    
    datetime_columns = list(set(datetime_columns + detected_datetime_columns))
        # text cols
    text_columns = []
    for col in object_columns:
        if col not in datetime_columns:
            avg_text_length = df[col].dropna().astype(str).str.len().mean()
    
            if avg_text_length > 30:
                text_columns.append(col)
                
        # const cols
    constant_columns = [
        col for col in all_columns
        if df[col].nunique(dropna=False) <= 1
    ]
    
    # category cols 
    excluded_from_categorical = set(
        datetime_columns
        + text_columns
        + boolean_columns
        + numeric_columns
    )
    
    categorical_columns = [
        col for col in all_columns
        if col not in excluded_from_categorical
    ]

    # data to table 
    dane = {
        "Metric": [
            "Numeric Columns",
            "Categorical Columns",
            "Text Columns",
            "Datetime Columns",
            "Boolean Columns",
            "Constant Columns",
        ],
        "Value": [
            len(numeric_columns),
            len(categorical_columns),
            len(text_columns),
            len(datetime_columns),
            len(boolean_columns),
            len(constant_columns),
        ],
        "Description": [
            "Columns with numeric values",
            "Columns with categorical values",
            "Columns with longer text values",
            "Columns detected as dates or timestamps",
            "Columns with true/false values",
            "Columns with only one unique value",
        ]
    }
    # markdown  
    title_md = mr.Markdown("# Dataset summary", key="title")
    col_type_md = mr.Markdown("### Column Type Summary", key="col_type")
    data_prev_md = mr.Markdown("### Data Preview", key="data_prev")
    # shape/miss/dup indicator
    ind_basic = mr.Indicator([
            mr.Indicator(value=row_count, label="Rows"),
            mr.Indicator(value=col_count, label="Columns"),
            mr.Indicator(value=duplicates_count, label="Duplicate Rows", delta=f"{duplicates_procent:.2f}%"),
            mr.Indicator(value=missing_count, label="Missing Values", delta=f"{missing_procent:.2f}%"),
        ])

In [ ]:
# basic indicators
if input_file.name is None:
    display(welcome_md)
else:
    display(title_md)

In [ ]:
# welcome message
if input_file.name is not None:
    display(ind_basic)
else: 
    clear_output(wait=False)

In [ ]:
# column type summary
if input_file.name is not None:
    display(col_type_md)
    tabelka = mr.Table(dane)
else: 
    clear_output(wait=False)

In [ ]:
# data preview
if input_file.name is not None:
    display(data_prev_md)
    report = TableReport(
        df,
        n_rows=15
    )
    
    display(report)
else: 
    clear_output(wait=False)